# LSTM from scratch: one forward step through the four gate equations

Companion to `notes.md`'s "From-scratch implementation" section. We implement, in plain
NumPy, exactly the gate equations from "LSTM Architecture (Forget, Input, Output
Gates)": forget gate, input gate + candidate, cell-state update, output gate + hidden
state. A single forward step (one timestep, going from $C_{t-1}, h_{t-1}$ to
$C_t, h_t$) is enough to see the mechanism — a second step is then taken with a
*different* toy input to see the cell state actually carrying information forward
rather than being overwritten, the property that distinguishes an LSTM from a vanilla
RNN.

In [1]:
import numpy as np

np.random.seed(0)
np.set_printoptions(precision=4, suppress=True)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

## A toy input and the four gate weight matrices

$x_t \in \mathbb{R}^3$, hidden/cell size $4$. Each gate takes the concatenation
$[h_{t-1}, x_t] \in \mathbb{R}^{4+3}$ per `notes.md`'s gate equations, so each weight
matrix $W_f, W_i, W_C, W_o$ has shape `(n_hidden, n_hidden + n_input)`.

In [2]:
n_input = 3
n_hidden = 4
concat_dim = n_hidden + n_input

Wf = np.random.randn(n_hidden, concat_dim) * 0.4
Wi = np.random.randn(n_hidden, concat_dim) * 0.4
WC = np.random.randn(n_hidden, concat_dim) * 0.4
Wo = np.random.randn(n_hidden, concat_dim) * 0.4
bf = np.zeros(n_hidden)
bi = np.zeros(n_hidden)
bC = np.zeros(n_hidden)
bo = np.zeros(n_hidden)

# Previous state, entering the cell at this timestep (t-1)
h_prev = np.zeros(n_hidden)
C_prev = np.zeros(n_hidden)

# Toy input at this timestep
x_t = np.array([1.0, 0.5, -0.5])
print("x_t =", x_t)
print("h_prev =", h_prev)
print("C_prev =", C_prev)

x_t = [ 1.   0.5 -0.5]
h_prev = [0. 0. 0. 0.]
C_prev = [0. 0. 0. 0.]


## Forget gate: how much of the old cell state to keep

$$f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$$

In [3]:
concat = np.concatenate([h_prev, x_t])  # [h_{t-1}, x_t]

f_t = sigmoid(Wf @ concat + bf)
print("f_t =", f_t)
print("(all values in [0, 1] -- a per-unit 'how much of C_prev to keep' fraction)")

f_t = [0.5894 0.6703 0.6142 0.3693]
(all values in [0, 1] -- a per-unit 'how much of C_prev to keep' fraction)


## Input gate + candidate: how much new information to write in

$$i_t = \sigma(W_i [h_{t-1}, x_t] + b_i), \qquad \tilde{C}_t = \tanh(W_C [h_{t-1}, x_t] + b_C)$$

In [4]:
i_t = sigmoid(Wi @ concat + bi)
C_tilde_t = np.tanh(WC @ concat + bC)

print("i_t       =", i_t)
print("C_tilde_t =", C_tilde_t)

i_t       = [0.3359 0.4883 0.4943 0.3627]
C_tilde_t = [-0.1764 -0.0067  0.1956  0.6415]


## Cell state update: retained old memory + gated new candidate

$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$

This is the **additive** update `notes.md` points to as the reason LSTM gradients
don't vanish the same way a vanilla RNN's do: $C_{t-1}$ is *added* into $C_t$ (scaled
by $f_t$), never fully replaced by a matrix multiplication the way a vanilla RNN's
$h_{t-1}$ is.

In [5]:
C_t = f_t * C_prev + i_t * C_tilde_t
print("C_t =", C_t)
print("\nCheck: since C_prev is all zeros here (first step), C_t reduces to i_t * C_tilde_t:")
print("i_t * C_tilde_t =", i_t * C_tilde_t)
print("matches C_t:", np.allclose(C_t, i_t * C_tilde_t))

C_t = [-0.0593 -0.0033  0.0967  0.2327]

Check: since C_prev is all zeros here (first step), C_t reduces to i_t * C_tilde_t:
i_t * C_tilde_t = [-0.0593 -0.0033  0.0967  0.2327]
matches C_t: True


## Output gate + hidden state: how much of the cell state to expose

$$o_t = \sigma(W_o [h_{t-1}, x_t] + b_o), \qquad h_t = o_t \odot \tanh(C_t)$$

In [6]:
o_t = sigmoid(Wo @ concat + bo)
h_t = o_t * np.tanh(C_t)

print("o_t =", o_t)
print("h_t =", h_t)

o_t = [0.4659 0.4819 0.4801 0.6867]
h_t = [-0.0276 -0.0016  0.0463  0.157 ]


## Wrapping the four equations into one reusable cell function

Same four equations as above, packaged as a function so a second timestep can reuse it
with the *same* weights (parameter sharing, exactly as in the RNN cell) but a new
input and the state just produced.

In [7]:
def lstm_cell_step(x_t, h_prev, C_prev, Wf, bf, Wi, bi, WC, bC, Wo, bo):
    concat = np.concatenate([h_prev, x_t])
    f_t = sigmoid(Wf @ concat + bf)
    i_t = sigmoid(Wi @ concat + bi)
    C_tilde_t = np.tanh(WC @ concat + bC)
    C_t = f_t * C_prev + i_t * C_tilde_t
    o_t = sigmoid(Wo @ concat + bo)
    h_t = o_t * np.tanh(C_t)
    return h_t, C_t, {"f_t": f_t, "i_t": i_t, "C_tilde_t": C_tilde_t, "o_t": o_t}

h_t_check, C_t_check, gates_check = lstm_cell_step(
    x_t, h_prev, C_prev, Wf, bf, Wi, bi, WC, bC, Wo, bo
)
assert np.allclose(h_t_check, h_t) and np.allclose(C_t_check, C_t)
print("function output matches the hand-computed step above: True")

function output matches the hand-computed step above: True


## A second step: watching the cell state carry information forward

Feed a *new* input $x_{t+1}$ using the $h_t, C_t$ just computed as the new
"previous" state. If the forget gate at this next step stays close to 1 for some
units, the corresponding entries of $C_t$ should survive largely intact into
$C_{t+1}$ rather than being wiped out -- the behavior a vanilla RNN's fully-overwritten
hidden state cannot exhibit.

In [8]:
x_t1 = np.array([-0.2, 0.1, 0.9])  # a different toy input

h_t1, C_t1, gates_t1 = lstm_cell_step(x_t1, h_t, C_t, Wf, bf, Wi, bi, WC, bC, Wo, bo)

print("forget gate at step t+1:", gates_t1["f_t"])
print("\nC_t   (before) =", C_t)
print("C_t+1 (after)  =", C_t1)
print("\ncontribution retained from C_t (f_t * C_t):", gates_t1["f_t"] * C_t)
print("contribution newly written in (i_t * C_tilde_t+1):", gates_t1["i_t"] * gates_t1["C_tilde_t"])
print("\nC_t+1 is exactly the sum of those two contributions:",
      np.allclose(C_t1, gates_t1["f_t"] * C_t + gates_t1["i_t"] * gates_t1["C_tilde_t"]))

forget gate at step t+1: [0.5726 0.4941 0.2752 0.5428]

C_t   (before) = [-0.0593 -0.0033  0.0967  0.2327]
C_t+1 (after)  = [-0.1657 -0.055  -0.1318 -0.1998]

contribution retained from C_t (f_t * C_t): [-0.0339 -0.0016  0.0266  0.1263]
contribution newly written in (i_t * C_tilde_t+1): [-0.1317 -0.0534 -0.1584 -0.3261]

C_t+1 is exactly the sum of those two contributions: True


## Cross-check against `tf.keras.layers.LSTMCell`

Keras's `LSTMCell` computes the same four equations, but stores its weights as one
concatenated kernel in gate order `[i, f, C, o]` rather than four separate matrices in
order `[f, i, C, o]`. Reassembling our weights into that layout and loading them lets
us confirm the from-scratch step matches the framework implementation exactly.

In [9]:
import tensorflow as tf

keras_cell = tf.keras.layers.LSTMCell(n_hidden, use_bias=True)
keras_cell.build((None, n_input))

# Keras LSTMCell kernel/recurrent_kernel/bias are each split [i, f, C, o] along axis -1,
# with kernel acting on x_t (shape n_input x 4*n_hidden) and recurrent_kernel acting on
# h_prev (shape n_hidden x 4*n_hidden). Our Wf/Wi/WC/Wo act on concat([h_prev, x_t]), so
# split each into its x-part and h-part and reassemble in Keras's order and shape.
def split_xh(W):
    W_h, W_x = W[:, :n_hidden], W[:, n_hidden:]
    return W_x.T, W_h.T  # -> (n_input, n_hidden), (n_hidden, n_hidden)

Wi_x, Wi_h = split_xh(Wi)
Wf_x, Wf_h = split_xh(Wf)
WC_x, WC_h = split_xh(WC)
Wo_x, Wo_h = split_xh(Wo)

kernel = np.concatenate([Wi_x, Wf_x, WC_x, Wo_x], axis=1).astype(np.float32)
recurrent_kernel = np.concatenate([Wi_h, Wf_h, WC_h, Wo_h], axis=1).astype(np.float32)
bias = np.concatenate([bi, bf, bC, bo]).astype(np.float32)

keras_cell.set_weights([kernel, recurrent_kernel, bias])

x_t_batch = x_t[np.newaxis, :].astype(np.float32)
h_prev_batch = h_prev[np.newaxis, :].astype(np.float32)
C_prev_batch = C_prev[np.newaxis, :].astype(np.float32)

(keras_h_t, [keras_h_t2, keras_C_t]) = keras_cell(x_t_batch, states=[h_prev_batch, C_prev_batch])

print("NumPy   h_t =", h_t)
print("Keras   h_t =", keras_h_t.numpy()[0])
print("NumPy   C_t =", C_t)
print("Keras   C_t =", keras_C_t.numpy()[0])
print("\nh_t match:", np.allclose(h_t, keras_h_t.numpy()[0], atol=1e-5))
print("C_t match:", np.allclose(C_t, keras_C_t.numpy()[0], atol=1e-5))

I0000 00:00:1787492294.825116  105332 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787492294.825476  105332 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1787492294.866877  105332 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1787492295.563915  105332 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787492295.564173  105332 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


NumPy   h_t = [-0.0276 -0.0016  0.0463  0.157 ]
Keras   h_t = [-0.0276 -0.0016  0.0463  0.157 ]
NumPy   C_t = [-0.0593 -0.0033  0.0967  0.2327]
Keras   C_t = [-0.0593 -0.0033  0.0967  0.2327]

h_t match: True
C_t match: True


E0000 00:00:1787492296.702602  105332 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1787492296.703051  105390 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1787492296.734049  105332 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


## Takeaway

Working through one LSTM step by hand shows the mechanism `notes.md` describes in
words: the forget gate and input gate jointly decide $C_t$ as a **weighted sum** of
old memory and new candidate information, not a full overwrite. The second-step
experiment above makes that additive property concrete — $C_{t+1}$ is exactly
`f_t * C_t + i_t * C_tilde_t`, so whatever the forget gate decides to keep passes
through by addition, not by repeated multiplication through a squashing non-linearity
the way a vanilla RNN's hidden state does. That is the structural reason gradients can
flow back through many such steps largely unattenuated.